# Predicción en batch con modelo de machine learning
Este notebook está diseñado para realizar predicciones en batch utilizando un modelo de machine learning previamente entrenado y registrado. El flujo típico incluye la carga de datos de entrada desde una tabla, la aplicación del modelo para generar predicciones y el almacenamiento de los resultados en una ubicación específica. Este proceso permite automatizar la inferencia sobre grandes volúmenes de datos de manera eficiente y reproducible.

## 1. Configuración e Importación de Librerías

In [0]:
# Celda 1: Recepción de Parámetros
schema_name = dbutils.widgets.text("schema_name", "")
schema_name = dbutils.widgets.get("schema_name")

silver_table = dbutils.widgets.text("silver_table", "")
silver_table = dbutils.widgets.get("silver_table")

In [0]:
import mlflow
from datetime import datetime

## 2. Cargar el conjunto de `datos`

In [0]:
# generar nombre completo de la tabla
def qname(table):
    return f"{schema_name}.{table}"

SILVER_FULL = qname(silver_table)
print("Tabla Silver:", SILVER_FULL)

In [0]:
# Leer datos de la tabla Bronze
lpn_prod = spark.table(SILVER_FULL)

display(lpn_prod.limit(20))

## 3. Cargar pipeline de transformaciones registrado

In [0]:
import mlflow

model_name = "pipe_transformer_registry"
client = mlflow.tracking.MlflowClient()
for mv in client.search_model_versions(f"name='{model_name}'"):
    print(f"Versión: {mv.version}, Estado: {mv.current_stage}, Run ID: {mv.run_id}")

In [0]:

pipe_uri = "models:/pipe_transformer_registry/1"
pipeModel = mlflow.spark.load_model(pipe_uri)


In [0]:
model_uri = "models:/modelo_xgb_registry/1"
model = mlflow.spark.load_model(model_uri)

## 4. Aplicar inferencia

In [0]:
lpn_prod_df = pipeModel.transform(lpn_prod)

In [0]:
predictions = model.transform(lpn_prod_df)

## 5. Guardar resultados

In [0]:
display(predictions.limit(10))

In [0]:
output_table_name = f"{schema_name}.predictions_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}"
predictions  \
    .select("LABEL_ZONE", "LABEL_ZONE_idx", "prediction") \
    .write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(output_table_name)